# Airline Passenger Satisfaction Prediction - End-to-End Pipeline

This notebook demonstrates the complete machine learning lifecycle for predicting **Airline Passenger Satisfaction**:
1. **Exploratory Data Analysis (EDA)** on 25,976 flight passenger records
2. **Data Cleaning & Handling Missing Delays**
3. **Feature Engineering** (`TotalDelay`, `CompositeServiceScore` across 14 service dimensions)
4. **Building Production Scikit-Learn Pipelines** (`StandardScaler` + `OneHotEncoder`)
5. **Multi-Model Training** (Logistic Regression, Random Forest, XGBoost)
6. **Model Evaluation & Leaderboard (> 85% Target Performance)**
7. **Confusion Matrix & ROC-AUC Curves**
8. **Production Pipeline Serialization**

In [1]:
import os
import sys
import yaml
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, RocCurveDisplay

# Ensure project root is in python path
sys.path.append(os.path.abspath('..'))

from src.preprocess import clean_raw_data, engineer_features, encode_target, build_preprocessor
from src.train import build_and_train_pipelines, save_model_pipeline
from src.evaluate import evaluate_single_pipeline, compare_models

sns.set_theme(style='whitegrid')

## 1. Load Airline Passenger Satisfaction Dataset

In [2]:
data_path = '../data/raw/test.csv'
df_raw = pd.read_csv(data_path)
print(f"Dataset dimensions: {df_raw.shape[0]:,} passenger records, {df_raw.shape[1]} features")
df_raw.head()

Dataset dimensions: 25,976 passenger records, 25 features


,Unnamed: 0,id,Gender,Customer Type,Age,Type of Travel,Class,Flight Distance,Inflight wifi service,Departure/Arrival time convenient,...,Inflight entertainment,On-board service,Leg room service,Baggage handling,Checkin service,Inflight service,Cleanliness,Departure Delay in Minutes,Arrival Delay in Minutes,satisfaction
0,0,19556,Female,Loyal Customer,52,Business travel,Eco,160,5,4,...,5,5,5,5,2,5,5,50,44.0,satisfied
1,1,90035,Female,Loyal Customer,36,Business travel,Business,2863,1,1,...,4,4,4,4,3,4,5,0,0.0,satisfied
2,2,12360,Male,disloyal Customer,20,Business travel,Eco,192,2,0,...,2,4,1,3,2,2,2,0,0.0,neutral or dissatisfied
3,3,77959,Male,Loyal Customer,44,Business travel,Business,3377,0,0,...,1,1,1,1,3,1,4,0,6.0,satisfied
4,4,36875,Female,Loyal Customer,49,Business travel,Eco,1182,2,3,...,2,2,2,2,4,2,4,0,20.0,satisfied


In [3]:
# Target class balance
plt.figure(figsize=(6, 3.5))
sns.countplot(data=df_raw, x='satisfaction', palette='Set2')
plt.title('Passenger Satisfaction Target Distribution')
plt.show()

print("Class Proportions:")
print(df_raw['satisfaction'].value_counts(normalize=True))

Class Proportions:
satisfaction
neutral or dissatisfied    0.561018
satisfied                  0.438982
Name: proportion, dtype: float64


C:\Users\rithv\AppData\Local\Temp\ipykernel_20804\3010995733.py:3: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.countplot(data=df_raw, x='satisfaction', palette='Set2')
C:\Users\rithv\AppData\Local\Temp\ipykernel_20804\3010995733.py:5: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 2. Exploratory Data Analysis: Key Satisfaction Drivers

In [4]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# 1. Satisfaction by Travel Class
sns.countplot(data=df_raw, x='Class', hue='satisfaction', ax=axes[0], palette='Blues_r')
axes[0].set_title('Satisfaction by Travel Class')

# 2. Satisfaction by Type of Travel
sns.countplot(data=df_raw, x='Type of Travel', hue='satisfaction', ax=axes[1], palette='Oranges_r')
axes[1].set_title('Satisfaction by Travel Type')

# 3. Online Boarding Rating vs Satisfaction
sns.countplot(data=df_raw, x='Online boarding', hue='satisfaction', ax=axes[2], palette='Greens_r')
axes[2].set_title('Satisfaction by Online Boarding Rating')

plt.tight_layout()
plt.show()

C:\Users\rithv\AppData\Local\Temp\ipykernel_20804\1162799190.py:16: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 3. Data Cleaning & Feature Engineering

In [5]:
# Load configuration
with open('../configs/config.yaml', 'r') as f:
    config = yaml.safe_load(f)

id_cols = config['data'].get('id_columns', ['Unnamed: 0', 'id'])
df_cleaned = clean_raw_data(df_raw, id_cols=id_cols)
df_featured = engineer_features(df_cleaned)

y = encode_target(df_featured['satisfaction'])
X = df_featured.drop(columns=['satisfaction'])

print("Engineered Features Preview:")
df_featured[['Flight Distance', 'TotalDelay', 'CompositeServiceScore', 'satisfaction']].head()

Engineered Features Preview:


,Flight Distance,TotalDelay,CompositeServiceScore,satisfaction
0,160,94.0,4.143,satisfied
1,2863,0.0,3.429,satisfied
2,192,0.0,2.143,neutral or dissatisfied
3,3377,6.0,1.786,satisfied
4,1182,20.0,2.643,satisfied


## 4. Train-Test Split & ColumnTransformer Preprocessor

In [6]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=config['data']['test_size'],
    random_state=config['data']['random_state'],
    stratify=y
)

num_cols = [c for c in config['features']['numerical'] if c in X.columns]
cat_cols = [c for c in config['features']['categorical'] if c in X.columns]

preprocessor = build_preprocessor(cat_cols, num_cols)
print(f"Pipeline configured: {len(num_cols)} numerical features, {len(cat_cols)} categorical features.")
print(f"Training set: {X_train.shape[0]:,} rows | Test set: {X_test.shape[0]:,} rows")

Pipeline configured: 20 numerical features, 4 categorical features.
Training set: 20,780 rows | Test set: 5,196 rows


## 5. Model Training & Comparison

In [7]:
trained_pipelines = build_and_train_pipelines(
    X_train=X_train,
    y_train=y_train,
    preprocessor=preprocessor,
    model_configs=config['models']
)

In [8]:
summary_df, all_metrics, best_model_name = compare_models(
    trained_pipelines=trained_pipelines,
    X_test=X_test,
    y_test=y_test
)

print("=== MODEL PERFORMANCE LEADERBOARD (PERCENTAGE FORMAT) ===")
summary_df

=== MODEL PERFORMANCE LEADERBOARD (PERCENTAGE FORMAT) ===


,Model,Accuracy (%),Precision (%),Recall (%),F1-Score (%),ROC-AUC (%)
0,XGBoost,96.09%,96.14%,94.91%,95.52%,99.40%
1,Random_Forest,95.40%,95.18%,94.30%,94.74%,99.07%
2,Logistic_Regression,87.22%,86.17%,84.44%,85.30%,92.68%


## 6. Confusion Matrix & ROC Curves

In [9]:
best_pipeline = trained_pipelines[best_model_name]

fig, ax = plt.subplots(1, 2, figsize=(14, 5))

# Confusion Matrix
cm = confusion_matrix(y_test, best_pipeline.predict(X_test))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax[0], xticklabels=['Dissatisfied', 'Satisfied'], yticklabels=['Dissatisfied', 'Satisfied'])
ax[0].set_title(f'Confusion Matrix ({best_model_name})')
ax[0].set_xlabel('Predicted Label')
ax[0].set_ylabel('True Label')

# ROC Curves
for name, pipe in trained_pipelines.items():
    RocCurveDisplay.from_estimator(pipe, X_test, y_test, name=name, ax=ax[1])
ax[1].set_title('ROC Curves Comparison')
ax[1].plot([0, 1], [0, 1], 'k--', alpha=0.7)

plt.tight_layout()
plt.show()

C:\Users\rithv\AppData\Local\Temp\ipykernel_20804\1423099662.py:19: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 7. Export Best Production Pipeline

In [10]:
model_export_path = '../models/best_model.joblib'
save_model_pipeline(best_pipeline, model_export_path)
print(f"Best model successfully saved to {model_export_path}")

Best model successfully saved to ../models/best_model.joblib
